# Fase 2 — Detección y picking (PhaseNet, EQTransformer)

Aplica los modelos preentrenados PhaseNet y EQTransformer (vía SeisBench) a la detección de eventos y al marcado de las ondas P y S sobre un subconjunto de trazas de STEAD, y evalúa su rendimiento por fase y tolerancia, con un análisis multi-semilla.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
import obspy
from obspy import UTCDateTime
from torchinfo import summary as TorchSummary

sys.path.append(os.path.abspath(""))

from utils.modelos import get_base_detector
from utils.data_loaders import STEADDataset


### Carga del modelo y exploración inicial

In [ ]:
RESULTS_DIR = os.path.join("..", "results")
FIGURES_DIR = os.path.join(RESULTS_DIR, "figures")

MODEL_NAME = "EQTransformer"   # "PhaseNet" o "EQTransformer"

model = get_base_detector(MODEL_NAME, pretrained = True)
model.eval()
print(f"Modelo cargado: {type(model).__name__}")
print(model)
TorchSummary(model, input_size = (1, 3, 6000))


In [ ]:
df = pd.read_csv(STEADDataset.CSV_PATH, low_memory = False)
df_sample = df[df["trace_category"] == "earthquake_local"].sample(2, random_state = 42)

with STEADDataset(df_sample) as dataset:
    tensor, trace_name = dataset[1]
    row = dataset.df.iloc[1]

# Metadata del CSV
p_real = int(row["p_arrival_sample"])
s_real = int(row["s_arrival_sample"])
start_time = UTCDateTime(row["trace_start_time"])
station = str(row["receiver_code"])
network = str(row["network_code"])
ch_type = str(row["receiver_type"])

# Construir obspy Stream desde el tensor (3, 6000): índices 0=E, 1=N, 2=Z
trazas = []
for i, comp in enumerate(["E", "N", "Z"]):
    tr = obspy.Trace(data = tensor[i].numpy().astype(np.float64))
    tr.stats.starttime = start_time
    tr.stats.delta = 0.01
    tr.stats.channel = ch_type + comp
    tr.stats.station = station
    tr.stats.network = network
    trazas.append(tr)
stream = obspy.Stream(trazas)

# Predicción de picks P y S con PhaseNet
output = model.classify(stream, P_threshold = 0.3, S_threshold = 0.3)
p_pred = [p.peak_time - start_time for p in output.picks if p.phase == "P"]
s_pred = [p.peak_time - start_time for p in output.picks if p.phase == "S"]
print(f"Picks predichos — P: {p_pred} s, S: {s_pred} s")
print(f"Llegadas reales  — P: {p_real / 100.0} s, S: {s_real / 100.0} s")

# Visualización: canal Z con llegadas reales vs. predichas
t = np.arange(6000) / 100.0
z = tensor[2].numpy()
z_norm = z / (np.max(np.abs(z)) + 1e-9)

fig, ax = plt.subplots(figsize = (12, 4))
ax.plot(t, z_norm, color = "black", linewidth = 0.7, label = "Canal Z")
ax.axvline(p_real / 100.0, color = "blue", linewidth = 2.0, label = "P real")
ax.axvline(s_real / 100.0, color = "red", linewidth = 2.0, label = "S real")
for pt in p_pred:
    ax.axvline(pt, color = "cyan", linewidth = 1.5, linestyle = "--", label = "P predicha")
for st in s_pred:
    ax.axvline(st, color = "orange", linewidth = 1.5, linestyle = "--", label = "S predicha")

handles, labels = ax.get_legend_handles_labels()
ax.legend(dict(zip(labels, handles)).values(), dict(zip(labels, handles)).keys(), loc = "upper right")
ax.set_xlabel("Tiempo (s)")
ax.set_ylabel("Amplitud (norm.)")
ax.set_title(f"{MODEL_NAME.lower()} — picks predichos vs. reales | {trace_name}")
plt.tight_layout()
plt.show()


### Análisis One-Seed

In [ ]:
SEED = 42
N_EVAL = 1000
P_THRESHOLD = 0.3
S_THRESHOLD = 0.3
TOLERANCIAS_P = [0.05, 0.10, 0.20, 0.50]
TOLERANCIAS_S = [0.10, 0.20, 0.50, 1.00]

df = pd.read_csv(STEADDataset.CSV_PATH, low_memory = False)
df_eval = (
    df[df["trace_category"] == "earthquake_local"]
    .dropna(subset = ["p_arrival_sample", "s_arrival_sample"])
    .sample(N_EVAL, random_state = SEED)
)

def construir_stream(tensor, row):
    start_time = UTCDateTime(row["trace_start_time"])
    ch_type = str(row["receiver_type"])
    trazas = []
    for i, comp in enumerate(["E", "N", "Z"]):
        tr = obspy.Trace(data = tensor[i].numpy().astype(np.float64))
        tr.stats.starttime = start_time
        tr.stats.delta = 0.01
        tr.stats.channel = ch_type + comp
        tr.stats.station = str(row["receiver_code"])
        tr.stats.network = str(row["network_code"])
        trazas.append(tr)
    return obspy.Stream(trazas), start_time

def evaluar_picks(all_picks, fase, tolerancia):
    """Empareja picks predichos con la llegada real para una fase y tolerancia.
    Devuelve dict con tp, fp, fn y array de residuos de los TPs."""
    tp = fp = fn = 0
    res_list = []
    for p_real, s_real, raw in all_picks:
        real = p_real if fase == "P" else s_real
        pred = np.array([t for ph, t in raw if ph == fase])

        if len(pred) > 0:
            errores = pred - real
            matched = np.zeros(len(pred), dtype = bool)
            valid = np.where(np.abs(errores) <= tolerancia)[0]

            if len(valid) > 0:
                best = valid[np.argmin(np.abs(errores[valid]))]
                tp += 1
                res_list.append(errores[best])
                matched[best] = True
            else:
                fn += 1

            fp += int(np.sum(~matched))
        else:
            fn += 1

    return {"tp": tp, "fp": fp, "fn": fn, "res": np.array(res_list)}

# Inferencia: una sola pasada por el modelo
all_picks = []
with STEADDataset(df_eval) as dataset:
    for idx in range(len(dataset)):
        tensor, _ = dataset[idx]
        row = dataset.df.iloc[idx]
        p_real = row["p_arrival_sample"] / 100.0
        s_real = row["s_arrival_sample"] / 100.0

        stream, start_time = construir_stream(tensor, row)
        output = model.classify(stream, P_threshold = P_THRESHOLD, S_threshold = S_THRESHOLD)

        raw = [(pick.phase, float(pick.peak_time - start_time)) for pick in output.picks]
        all_picks.append((p_real, s_real, raw))

# Evaluar todas las tolerancias de cada fase
metrics = {
    "P": {tol: evaluar_picks(all_picks, "P", tol) for tol in TOLERANCIAS_P},
    "S": {tol: evaluar_picks(all_picks, "S", tol) for tol in TOLERANCIAS_S},
}

print(f"Loop completado — {N_EVAL} trazas evaluadas (Pth = {P_THRESHOLD}, Sth = {S_THRESHOLD}).")

In [ ]:
def calcular_fila(d):
    """Devuelve una fila de métricas formateadas para la tabla."""
    tp, fp, fn, res = d["tp"], d["fp"], d["fn"], d["res"]
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    if len(res) > 0:
        residuo = f"{np.mean(res) * 1000:.1f} ± {np.std(res) * 1000:.1f}"
        mae     = f"{np.mean(np.abs(res)) * 1000:.1f}"
        rmse    = f"{np.sqrt(np.mean(res ** 2)) * 1000:.1f}"
        p50     = f"{100 * np.mean(np.abs(res) < 0.05):.1f}%"
        p100    = f"{100 * np.mean(np.abs(res) < 0.10):.1f}%"
    else:
        residuo = mae = rmse = p50 = p100 = "-"
    return [f"{precision:.3f}", f"{recall:.3f}", f"{f1:.3f}", residuo, mae, rmse, p50, p100, str(len(res))]

COL_LABELS = ["Prec.", "Rec.", "F1", "Res. ± std (ms)", "MAE (ms)", "RMSE (ms)", "≤ 50 ms", "≤ 100 ms", "n"]

# Sufijos comunes para títulos y nombres de archivo
TH_TAG   = f"pth{P_THRESHOLD:.2f}_sth{S_THRESHOLD:.2f}"
TH_LABEL = f"Pth = {P_THRESHOLD:.2f}, Sth = {S_THRESHOLD:.2f}"

# --- Tabla de métricas: P y S apilados verticalmente — 300 DPI ---
n_filas = max(len(TOLERANCIAS_P), len(TOLERANCIAS_S))
fig, axs = plt.subplots(2, 1, figsize = (12, n_filas * 1.4 + 2))
for ax, fase, tolerancias_fase, color in zip(
    axs, ["P", "S"], [TOLERANCIAS_P, TOLERANCIAS_S], ["darkslategray", "darkred"]
):
    filas      = [calcular_fila(metrics[fase][tol]) for tol in tolerancias_fase]
    row_labels = [f"±{tol * 1000:.0f} ms" for tol in tolerancias_fase]

    ax.axis("off")
    t = ax.table(
        cellText  = filas,
        rowLabels = row_labels,
        colLabels = COL_LABELS,
        loc       = "center",
        cellLoc   = "center",
    )
    t.auto_set_font_size(False)
    t.set_fontsize(9)
    t.scale(1, 1.8)
    ax.set_title(f"Onda {fase} — {MODEL_NAME} (N = {N_EVAL}, seed = {SEED}, {TH_LABEL})", fontsize = 11, fontweight = "bold", color = color)

plt.suptitle(f"Métricas de picking por tolerancia ({TH_LABEL})", fontsize = 13)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/eval_{MODEL_NAME.lower()}_metricas_{TH_TAG}.png", dpi = 300, bbox_inches = "tight")
plt.show()

# --- Histogramas de residuos: 2 filas × N tolerancias — 300 DPI ---
n_cols = max(len(TOLERANCIAS_P), len(TOLERANCIAS_S))
fig, axs = plt.subplots(2, n_cols, figsize = (4 * n_cols, 7))
for row, (fase, tolerancias_fase, color) in enumerate(
    [("P", TOLERANCIAS_P, "steelblue"), ("S", TOLERANCIAS_S, "tomato")]
):
    for col, tol in enumerate(tolerancias_fase):
        ax  = axs[row, col]
        res = metrics[fase][tol]["res"]
        ax.hist(res * 1000, bins = 40, color = color, alpha = 0.75, edgecolor = "black", linewidth = 0.4)
        ax.axvline(0, color = "black", linewidth = 1.2, linestyle = "--")
        ax.set_xlabel("Residuo (ms)")
        ax.set_ylabel("Frecuencia")
        ax.set_title(f"Onda {fase} — tol ±{tol * 1000:.0f} ms (n = {len(res)})")
    for col in range(len(tolerancias_fase), n_cols):
        axs[row, col].axis("off")

plt.suptitle(f"Residuos picking {MODEL_NAME} — STEAD (N = {N_EVAL}, seed = {SEED}, {TH_LABEL})", fontsize = 13)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/eval_{MODEL_NAME.lower()}_residuos_{TH_TAG}.png", dpi = 300, bbox_inches = "tight")
plt.show()

Los resultados se guardan en formato pickle para su uso posterior.

In [ ]:
import pickle

# Persistencia de las métricas de este modelo en disco para poder cargar
# luego ambos (PhaseNet y EQTransformer) y construir la tabla comparativa.
TH_TAG = f"pth{P_THRESHOLD:.2f}_sth{S_THRESHOLD:.2f}"

payload = {
    "model_name":    MODEL_NAME,
    "seed":          SEED,
    "n_eval":        N_EVAL,
    "p_threshold":   P_THRESHOLD,
    "s_threshold":   S_THRESHOLD,
    "tolerancias_p": TOLERANCIAS_P,
    "tolerancias_s": TOLERANCIAS_S,
    "metrics":       metrics,
}

filename = f"metrics_{MODEL_NAME.lower()}_seed{SEED}_N{N_EVAL}_{TH_TAG}.pkl"
filepath = os.path.join(RESULTS_DIR, filename)

with open(filepath, "wb") as f:
    pickle.dump(payload, f)

print(f"Métricas guardadas en: {filepath}")

In [ ]:
import pickle

TH_TAG   = f"pth{P_THRESHOLD:.2f}_sth{S_THRESHOLD:.2f}"
TH_LABEL = f"Pth = {P_THRESHOLD:.2f}, Sth = {S_THRESHOLD:.2f}"

# Cargar métricas de ambos modelos desde disco
nombres = ["phasenet", "eqtransformer"]
datos = {}
for nombre in nombres:
    path = os.path.join(RESULTS_DIR, f"metrics_{nombre}_seed{SEED}_N{N_EVAL}_{TH_TAG}.pkl")
    with open(path, "rb") as f:
        datos[nombre] = pickle.load(f)
    assert datos[nombre]["seed"] == SEED, f"Seed mismatch: {nombre}"
    assert datos[nombre]["n_eval"] == N_EVAL, f"N_EVAL mismatch: {nombre}"
    assert datos[nombre]["p_threshold"] == P_THRESHOLD, f"P_threshold mismatch: {nombre}"
    assert datos[nombre]["s_threshold"] == S_THRESHOLD, f"S_threshold mismatch: {nombre}"

display_names = {n: datos[n]["model_name"] for n in nombres}

# --- Comparativa: 2 filas (P, S) × 2 cols (PhaseNet, EQTransformer) — 300 DPI ---
n_filas = max(len(TOLERANCIAS_P), len(TOLERANCIAS_S))
fig, axs = plt.subplots(2, 2, figsize = (22, n_filas * 1.4 + 2))
colores = {"P": "darkslategray", "S": "darkred"}

for row_idx, (fase, tolerancias_fase) in enumerate(
    [("P", TOLERANCIAS_P), ("S", TOLERANCIAS_S)]
):
    for col_idx, nombre in enumerate(nombres):
        ax = axs[row_idx, col_idx]
        m  = datos[nombre]["metrics"]
        filas      = [calcular_fila(m[fase][tol]) for tol in tolerancias_fase]
        row_labels = [f"±{tol * 1000:.0f} ms" for tol in tolerancias_fase]

        ax.axis("off")
        t = ax.table(
            cellText  = filas,
            rowLabels = row_labels,
            colLabels = COL_LABELS,
            loc       = "center",
            cellLoc   = "center",
        )
        t.auto_set_font_size(False)
        t.set_fontsize(9)
        t.scale(1, 1.8)
        ax.set_title(
            f"Onda {fase} — {display_names[nombre]} (N = {N_EVAL}, seed = {SEED})",
            fontsize = 11, fontweight = "bold", color = colores[fase]
        )

plt.suptitle(
    f"Comparativa PhaseNet vs EQTransformer — {TH_LABEL}",
    fontsize = 14, fontweight = "bold"
)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/eval_comparativa_metricas_{TH_TAG}.png", dpi = 300, bbox_inches = "tight")
plt.show()

### Análisis Multi-Seed

In [ ]:
import pickle
import time

P_THRESHOLD=S_THRESHOLD=0.1

# --- Evaluación multi-semilla: media ± std + tests estadísticos ---
SEEDS_MULTI  = [42, 0, 1, 7, 13]
N_EVAL_MULTI = 200
MODEL_NAMES  = ["PhaseNet", "EQTransformer"]

# CSV cargado una sola vez (filtrado a earthquake_local con llegadas P/S)
df_full = pd.read_csv(STEADDataset.CSV_PATH, low_memory = False)
df_full = (df_full[df_full["trace_category"] == "earthquake_local"]
                  .dropna(subset = ["p_arrival_sample", "s_arrival_sample"]))

resultados_multi = {}
for nombre in MODEL_NAMES:
    print(f"\n=== {nombre} ===")
    modelo = get_base_detector(nombre, pretrained = True)
    modelo.eval()

    por_seed = {}
    for seed in SEEDS_MULTI:
        t0 = time.time()
        df_eval = df_full.sample(N_EVAL_MULTI, random_state = seed)

        all_picks = []
        with STEADDataset(df_eval) as ds:
            for idx in range(len(ds)):
                tensor, _ = ds[idx]
                row = ds.df.iloc[idx]
                p_real = row["p_arrival_sample"] / 100.0
                s_real = row["s_arrival_sample"] / 100.0
                stream, start_time = construir_stream(tensor, row)
                output = modelo.classify(stream, P_threshold = P_THRESHOLD, S_threshold = S_THRESHOLD)
                raw = [(p.phase, float(p.peak_time - start_time)) for p in output.picks]
                all_picks.append((p_real, s_real, raw))

        por_seed[seed] = {
            "P": {tol: evaluar_picks(all_picks, "P", tol) for tol in TOLERANCIAS_P},
            "S": {tol: evaluar_picks(all_picks, "S", tol) for tol in TOLERANCIAS_S},
        }
        print(f"  seed = {seed}: {N_EVAL_MULTI} trazas en {time.time() - t0:.1f}s")

    resultados_multi[nombre] = por_seed

# Persistencia: un único archivo con todos los modelos y seeds
TH_TAG_MS = f"pth{P_THRESHOLD:.2f}_sth{S_THRESHOLD:.2f}"
filename  = f"metrics_multiseed_N{N_EVAL_MULTI}_{len(SEEDS_MULTI)}seeds_{TH_TAG_MS}.pkl"
filepath  = os.path.join(RESULTS_DIR, filename)
with open(filepath, "wb") as f:
    pickle.dump({
        "seeds":         SEEDS_MULTI,
        "n_eval":        N_EVAL_MULTI,
        "p_threshold":   P_THRESHOLD,
        "s_threshold":   S_THRESHOLD,
        "tolerancias_p": TOLERANCIAS_P,
        "tolerancias_s": TOLERANCIAS_S,
        "resultados":    resultados_multi,
    }, f)
print(f"\nMulti-seed guardado en: {filepath}")

In [ ]:
from scipy.stats import wilcoxon

os.makedirs(FIGURES_DIR, exist_ok = True)

# --- Carga del multi-seed desde disco ---
TH_TAG_MS = f"pth{P_THRESHOLD:.2f}_sth{S_THRESHOLD:.2f}"
TH_LABEL  = f"Pth = {P_THRESHOLD:.2f}, Sth = {S_THRESHOLD:.2f}"
filename  = f"metrics_multiseed_N{N_EVAL_MULTI}_{len(SEEDS_MULTI)}seeds_{TH_TAG_MS}.pkl"
filepath  = os.path.join(RESULTS_DIR, filename)
with open(filepath, "rb") as f:
    multi = pickle.load(f)
resultados   = multi["resultados"]
seeds_loaded = multi["seeds"]

def metricas_eval(d):
    """Devuelve [prec, rec, f1, mae_ms, rmse_ms] para un eval (una seed, una tol)."""
    tp, fp, fn, res = d["tp"], d["fp"], d["fn"], d["res"]
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    mae  = float(np.mean(np.abs(res)) * 1000)         if len(res) > 0 else 0.0
    rmse = float(np.sqrt(np.mean(res ** 2)) * 1000)    if len(res) > 0 else 0.0
    return [prec, rec, f1, mae, rmse]

COL_LABELS_MS = ["Prec.", "Rec.", "F1", "MAE (ms)", "RMSE (ms)"]
DECIMALES_MS  = [3, 3, 3, 1, 1]

def fila_mean_std(modelo, fase, tol):
    M = np.array([metricas_eval(resultados[modelo][s][fase][tol]) for s in seeds_loaded])
    means, stds = M.mean(axis = 0), M.std(axis = 0)
    return [f"{means[i]:.{DECIMALES_MS[i]}f} ± {stds[i]:.{DECIMALES_MS[i]}f}" for i in range(len(means))]

# --- Tabla 2×2 (P/S × PhaseNet/EQT): media ± std — 300 DPI ---
n_filas = max(len(multi["tolerancias_p"]), len(multi["tolerancias_s"]))
fig, axs = plt.subplots(2, 2, figsize = (22, n_filas * 1.4 + 2))
colores = {"P": "darkslategray", "S": "darkred"}

for row_idx, (fase, tolerancias_fase) in enumerate(
    [("P", multi["tolerancias_p"]), ("S", multi["tolerancias_s"])]
):
    for col_idx, modelo in enumerate(MODEL_NAMES):
        ax    = axs[row_idx, col_idx]
        filas = [fila_mean_std(modelo, fase, tol) for tol in tolerancias_fase]
        row_labels = [f"±{tol * 1000:.0f} ms" for tol in tolerancias_fase]

        ax.axis("off")
        t = ax.table(
            cellText  = filas,
            rowLabels = row_labels,
            colLabels = COL_LABELS_MS,
            loc       = "center",
            cellLoc   = "center",
        )
        t.auto_set_font_size(False)
        t.set_fontsize(9)
        t.scale(1, 1.8)
        ax.set_title(
            f"Onda {fase} — {modelo} (N = {multi['n_eval']}, {len(seeds_loaded)} seeds)",
            fontsize = 11, fontweight = "bold", color = colores[fase]
        )

plt.suptitle(
    f"Multi-seed: media ± std — {TH_LABEL}",
    fontsize = 14, fontweight = "bold"
)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, f"eval_multiseed_metricas_{TH_TAG_MS}.png"), dpi = 300, bbox_inches = "tight")
plt.show()

# --- Wilcoxon pareado: F1 PhaseNet vs EQTransformer por (fase, tol) ---
filas_w = []
for fase, tolerancias_fase in [("P", multi["tolerancias_p"]), ("S", multi["tolerancias_s"])]:
    for tol in tolerancias_fase:
        f1_pn  = [metricas_eval(resultados["PhaseNet"][s][fase][tol])[2]      for s in seeds_loaded]
        f1_eqt = [metricas_eval(resultados["EQTransformer"][s][fase][tol])[2] for s in seeds_loaded]
        try:
            stat, p = wilcoxon(f1_pn, f1_eqt)
            sig = "*" if p < 0.05 else ""
            filas_w.append([
                fase,
                f"±{tol * 1000:.0f} ms",
                f"{np.mean(f1_pn):.3f} ± {np.std(f1_pn):.3f}",
                f"{np.mean(f1_eqt):.3f} ± {np.std(f1_eqt):.3f}",
                f"{p:.4f}",
                sig,
                p < 0.05,
            ])
        except Exception as e:
            filas_w.append([fase, f"±{tol * 1000:.0f} ms", "-", "-", str(e), "", False])

COL_LABELS_W = ["Fase", "Tolerancia", "PhaseNet F1", "EQTransformer F1", "p-value", "Sig."]

fig, ax = plt.subplots(figsize = (16, len(filas_w) * 0.55 + 2.5))
ax.axis("off")
t = ax.table(
    cellText  = [fila[:6] for fila in filas_w],
    colLabels = COL_LABELS_W,
    loc       = "center",
    cellLoc   = "center",
)
t.auto_set_font_size(False)
t.set_fontsize(10)
t.scale(1, 1.8)

# Resaltar filas significativas en verde claro
for i, fila in enumerate(filas_w):
    if fila[6]:
        for j in range(len(COL_LABELS_W)):
            t[(i + 1, j)].set_facecolor("#d4edda")

ax.set_title(
    f"Wilcoxon pareado F1 — PhaseNet vs EQTransformer "
    f"({len(seeds_loaded)} seeds, N = {multi['n_eval']}, {TH_LABEL})\n"
    f"H0: distribuciones de F1 iguales · * = p < 0.05 (filas resaltadas)",
    fontsize = 12, fontweight = "bold"
)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, f"eval_multiseed_metricas_wilcoxon_{TH_TAG_MS}.png"), dpi = 300, bbox_inches = "tight")
plt.show()

# Print en consola para revisión rápida
print("=== Wilcoxon pareado: F1 PhaseNet vs EQTransformer ===")
print(f"({len(seeds_loaded)} seeds — H0: las distribuciones de F1 son iguales)")
print(f"{'Fase':>5} {'Tol':>9} | {'PhaseNet F1':>17} | {'EQT F1':>17} | {'p-value':>9}")
print("-" * 72)
for fila in filas_w:
    sig = fila[5]
    print(f"{fila[0]:>5} {fila[1]:>9} | {fila[2]:>17} | {fila[3]:>17} | {fila[4]:>8} {sig}")